# Stage 1A: corrected modern SwinJSCC port, K=256 VQ, and AWGN evaluation

This is the only active Stage 1A notebook. It clones a pinned project revision and invokes the modular repository implementation. It is a runner, not a second source of truth.

Research status: this is a corrected modern PyTorch port of the pinned SwinJSCC source, not a paper-exact replication.

## 1. Run controls and pinned inputs

Set `PROJECT_GIT_URL` after pushing the local Git history. Keep `PROJECT_REF` unchanged: it is the exact source revision the experiment uses. Kaggle Internet must be enabled for the two Git clones.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

PROJECT_GIT_URL = 'https://github.com/REPLACE_WITH_YOUR_ACCOUNT/SemanticSchedulerEnd2End.git'
PROJECT_REF = 'SET_TO_COMMITTED_SOURCE_REVISION'
UPSTREAM_SWINJSCC_COMMIT = 'a6d0e6da53548976acbe9317839a077ef31f190f'
RUN_TRAINING = True
RUN_EVALUATION = True
RUN_FULL_JPEG_QUALITY_SWEEP = True
STAGE1_OUTPUT = Path('/kaggle/working/stage1a_complete_output')
PROJECT_DIR = Path('/kaggle/working/SemanticSchedulerEnd2End')
TRAIN_ROOT = Path('/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr/Training/HR/hr_images')
VALIDATION_ROOT = Path('/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr/Validation/HR/hr_images')
KODAK_ROOT = Path('/kaggle/input/datasets/sherylmehta/kodak-dataset')
BASE_EPOCHS = 200
SARA_EPOCHS = 300
BATCH_SIZE_PER_GPU = 8
EARLY_STOPPING_PATIENCE = 30
KMEANS_FIT_IMAGES = 512
VQ_EVALUATION_RATE = 96
JPEG_QUALITY_GRID = (20, 30, 40, 50, 60, 70, 80, 90)

def run_command(*arguments: str, cwd: Path | None = None) -> None:
    print('+', ' '.join(arguments))
    subprocess.run(list(arguments), cwd=cwd, check=True)

if 'REPLACE_WITH_YOUR_ACCOUNT' in PROJECT_GIT_URL:
    raise RuntimeError('Set PROJECT_GIT_URL to the pushed repository remote before running this notebook.')
if PROJECT_REF.startswith('SET_'):
    raise RuntimeError('PROJECT_REF must be a real immutable Git commit before running this notebook.')


## 2. Clone the pinned project and install dependencies without replacing Kaggle PyTorch

In [ ]:
import torch

torch_before = str(torch.__version__)
if PROJECT_DIR.exists():
    raise RuntimeError(f'Refusing to reuse an old project directory: {PROJECT_DIR}')
run_command('git', 'clone', '--no-checkout', PROJECT_GIT_URL, str(PROJECT_DIR))
run_command('git', 'checkout', '--detach', PROJECT_REF, cwd=PROJECT_DIR)
resolved_ref = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
if resolved_ref != PROJECT_REF:
    raise RuntimeError(f'Pinned project mismatch: expected {PROJECT_REF}, got {resolved_ref}')
run_command(sys.executable, 'scripts/bootstrap_swinjscc.py', cwd=PROJECT_DIR)
upstream_ref = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR / 'external' / 'SwinJSCC', text=True).strip()
if upstream_ref != UPSTREAM_SWINJSCC_COMMIT:
    raise RuntimeError(f'Pinned SwinJSCC mismatch: expected {UPSTREAM_SWINJSCC_COMMIT}, got {upstream_ref}')
run_command(sys.executable, '-m', 'pip', 'install', '--no-deps', '-r', 'environment/requirements-kaggle.lock', cwd=PROJECT_DIR)
run_command(sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', '.[metrics]', cwd=PROJECT_DIR)
if str(torch.__version__) != torch_before:
    raise RuntimeError("The notebook must not replace Kaggle's CUDA-enabled PyTorch build.")
os.chdir(PROJECT_DIR)
STAGE1_OUTPUT.mkdir(parents=True, exist_ok=False)


## 3. Strict runtime and real-data gates

In [ ]:
from EncDecPipeline.Models.SwinJSCC.stage1_runner import (
    evaluate_jpeg_over_digital_awgn,
    evaluate_native_swinjscc,
    evaluate_vq_noiseless_ablation,
    evaluate_vq_over_digital_awgn,
    export_final_stage1_artifacts,
    fit_k256_codebook,
    resolve_required_datasets,
    run_base_then_sara_training,
    verify_kaggle_stage1a_runtime,
)
from EncDecPipeline.Models.SwinJSCC.swin_config import SwinJSCCConfig
from EncDecPipeline.Models.SwinJSCC.trainer import RealImageDataset
from EncDecPipeline.Models.SwinJSCC.training_utils import mse_loss, wrap_data_parallel
from Evaluation.Reporting.plotting import plot_quality_vs_channel_uses
from Evaluation.Reporting.report_builder import write_run_report
from Evaluation.Reporting.result_table import write_result_table
from utils.seed import set_seed

set_seed(20260906)
runtime = verify_kaggle_stage1a_runtime()
data_provenance = resolve_required_datasets(TRAIN_ROOT, VALIDATION_ROOT, KODAK_ROOT)
config = SwinJSCCConfig()
assert config.rate_grid == (32, 64, 96, 128, 192)
assert config.snr_db_grid == (1, 4, 7, 10, 13)
assert config.channel_type == 'awgn' and config.objective == 'mse'
(STAGE1_OUTPUT / 'startup.json').write_text(json.dumps({'runtime': runtime, 'data_provenance': data_provenance}, indent=2), encoding='utf-8')
print(json.dumps(runtime, indent=2))


## 4. Two-T4 preflight: one real batch forward/backward and one-image overfit sanity check

In [ ]:
from EncDecPipeline.Models.SwinJSCC.trainer import build_base_then_sara
from EncDecPipeline.Models.SwinJSCC.training_utils import verify_two_t4_data_parallel

plan = verify_two_t4_data_parallel()
preflight_adapter, _ = build_base_then_sara(config, 'external/SwinJSCC')
preflight_model = wrap_data_parallel(preflight_adapter.build_training_module(), plan)
preflight_optimizer = torch.optim.AdamW(preflight_model.parameters(), lr=1e-4)
preflight_dataset = RealImageDataset(TRAIN_ROOT, crop_size=256, training=True, seed=20260906)
real_image = preflight_dataset[0].unsqueeze(0).cuda()
preflight_model.train()
losses = []
for step in range(8):
    preflight_optimizer.zero_grad(set_to_none=True)
    reconstruction, _ = preflight_model(real_image, 10, config.base_fixed_c)
    loss = mse_loss(real_image, reconstruction)
    loss.backward()
    preflight_optimizer.step()
    losses.append(float(loss.detach().cpu()))
if not all(torch.isfinite(torch.tensor(losses))):
    raise RuntimeError(f'Non-finite real-image preflight loss: {losses}')
(STAGE1_OUTPUT / 'preflight.json').write_text(json.dumps({'losses': losses, 'note': 'finite-loss and two-GPU forward/backward check; not a training-quality claim'}, indent=2), encoding='utf-8')
del preflight_model, preflight_adapter, preflight_optimizer
torch.cuda.empty_cache()


## 5. Train Base SwinJSCC then transfer to SA+RA, checkpoint/resume, and export tensor-state artifacts

In [ ]:
if RUN_TRAINING:
    adapter, training_summary = run_base_then_sara_training(
        config=config,
        upstream_root='external/SwinJSCC',
        train_root=TRAIN_ROOT,
        validation_root=VALIDATION_ROOT,
        output_dir=STAGE1_OUTPUT,
        base_epochs=BASE_EPOCHS,
        sara_epochs=SARA_EPOCHS,
        batch_size_per_gpu=BATCH_SIZE_PER_GPU,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
    )
    export_final_stage1_artifacts(
        adapter,
        STAGE1_OUTPUT / 'models' / 'SwinJSCC',
        data_provenance=data_provenance,
        training_summary=training_summary,
    )
else:
    raise RuntimeError('Stage 1A evaluation requires a model trained in this notebook or explicitly loaded through the manifest.')


## 6. Fit and save the K=256 MiniBatchKMeans codebook from real SA+RA latents

In [ ]:
vq = fit_k256_codebook(adapter, TRAIN_ROOT, rate=VQ_EVALUATION_RATE, max_images=KMEANS_FIT_IMAGES)
codebook_path = vq.save(STAGE1_OUTPUT / 'models' / 'Codebooks' / 'swinjscc_tx_k256.joblib')
print({'codebook': str(codebook_path), 'codebook_hash': vq.content_hash()})


## 7. Evaluate native JSCC, the separate no-channel VQ ablation, VQ packet transport, and JPEG/5G-LDPC/QPSK over AWGN

In [ ]:
if RUN_EVALUATION:
    rows = []
    rows.extend(evaluate_native_swinjscc(adapter, KODAK_ROOT, STAGE1_OUTPUT, config.snr_db_grid, config.rate_grid))
    rows.extend(evaluate_vq_noiseless_ablation(adapter, vq, KODAK_ROOT, STAGE1_OUTPUT, rate=VQ_EVALUATION_RATE))
    rows.extend(evaluate_vq_over_digital_awgn(adapter, vq, KODAK_ROOT, STAGE1_OUTPUT, config.snr_db_grid, rate=VQ_EVALUATION_RATE))
    jpeg_qualities = JPEG_QUALITY_GRID if RUN_FULL_JPEG_QUALITY_SWEEP else (50,)
    for quality in jpeg_qualities:
        rows.extend(evaluate_jpeg_over_digital_awgn(KODAK_ROOT, STAGE1_OUTPUT, config.snr_db_grid, quality))
    metrics_path = write_result_table(rows, STAGE1_OUTPUT / 'metrics' / 'stage1a_metrics.csv')
    successful_rows = [row for row in rows if row.get('psnr') is not None and row['method'] != 'vq_noiseless_ablation']
    plot_quality_vs_channel_uses(successful_rows, STAGE1_OUTPUT / 'plots' / 'quality_vs_channel_uses.png')
    write_run_report(
        {'project_ref': PROJECT_REF, 'upstream_commit': UPSTREAM_SWINJSCC_COMMIT, 'metrics_csv': str(metrics_path), 'rows': len(rows), 'corrected_port': True},
        STAGE1_OUTPUT / 'reports' / 'stage1a_report.md',
    )


## 8. Final output audit

The expected output directory contains tensor-state artifacts, a manifest, K=256 codebook, real-data provenance, training/checkpoint summaries, metrics, plots, reconstructions, and report. Failed digital frames remain in the table with null PSNR/SSIM.

In [ ]:
expected = [
    STAGE1_OUTPUT / 'models' / 'SwinJSCC' / 'stage1_swinjscc_full.pt',
    STAGE1_OUTPUT / 'models' / 'SwinJSCC' / 'stage1_swinjscc_encoder.pt',
    STAGE1_OUTPUT / 'models' / 'SwinJSCC' / 'stage1_swinjscc_decoder.pt',
    STAGE1_OUTPUT / 'models' / 'SwinJSCC' / 'stage1_manifest.json',
    STAGE1_OUTPUT / 'models' / 'Codebooks' / 'swinjscc_tx_k256.joblib',
    STAGE1_OUTPUT / 'metrics' / 'stage1a_metrics.csv',
]
missing = [str(path) for path in expected if not path.exists()]
if missing:
    raise RuntimeError(f'Stage 1A output audit failed: {missing}')
print('Stage 1A Kaggle run complete:', STAGE1_OUTPUT)
